# 04 — Final Evaluation: Next-Day Air Quality (Delhi)

This notebook evaluates the saved Phase C regression predictions and independently reruns the three classification models on the same chronological test period. It writes final metric tables and figures under `results/`.

**Evaluation period:** target dates 2020-01-01 through 2020-07-01. The source dataset ends on 2020-07-01, so the target date is formed as source date + 1 day and the test mask is capped at 2020-07-01. No random shuffle is used.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve)

ROOT = Path.cwd()
# Supports running from the repository root or the notebooks directory.
if not (ROOT / 'results/phase_c/delhi_phasec_features.csv').exists() and (ROOT.parent / 'results/phase_c/delhi_phasec_features.csv').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
PHASE_C = RESULTS / 'phase_c'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
features_path = PHASE_C / 'delhi_phasec_features.csv'
reg_predictions_path = PHASE_C / 'regression_predictions.csv'
if not features_path.exists():
    raise FileNotFoundError(f'Missing {features_path}. Run 02-preprocessing first and place delhi_phasec_features.csv there.')
if not reg_predictions_path.exists():
    raise FileNotFoundError(f'Missing {reg_predictions_path}. Run 03-models first and place its results in results/phase_c/.')
features_df = pd.read_csv(features_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
reg_pred = pd.read_csv(reg_predictions_path, parse_dates=['target_date'])
assert features_df['Date'].is_unique and features_df['Date'].is_monotonic_increasing
print('Feature data:', features_df.shape, features_df.Date.min().date(), 'to', features_df.Date.max().date())
print('Saved regression predictions:', reg_pred.shape, reg_pred.target_date.min().date(), 'to', reg_pred.target_date.max().date())

## 1. Regression metrics and persistence comparison

The persistence forecast is `tomorrow PM2.5 = today's PM2.5`. All regression models and the baseline must be scored on the exact same saved target dates. Lower MAE/RMSE is better; higher R² is better.

In [ ]:
reg_rows = []
actual = reg_pred['actual_PM2.5'].to_numpy()
for model_name in ['Persistence', 'Linear Regression', 'Ridge', 'Lasso', 'Random Forest Regressor']:
    pred = reg_pred[model_name].to_numpy()
    reg_rows.append({
        'task': 'regression', 'model': model_name, 'test_rows': len(actual),
        'MAE': mean_absolute_error(actual, pred),
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'R2': r2_score(actual, pred)
    })
reg_metrics = pd.DataFrame(reg_rows).sort_values('RMSE').reset_index(drop=True)
base = reg_metrics.loc[reg_metrics.model.eq('Persistence')].iloc[0]
reg_metrics['MAE_improvement_vs_persistence_pct'] = 100 * (base.MAE - reg_metrics.MAE) / base.MAE
reg_metrics['RMSE_improvement_vs_persistence_pct'] = 100 * (base.RMSE - reg_metrics.RMSE) / base.RMSE
reg_metrics.to_csv(RESULTS / 'regression_metrics_final.csv', index=False)
reg_metrics.round(4)

In [ ]:
# Confirm saved baseline equals the observed PM2.5 on the day before each target date.
lookup = features_df.set_index('Date')['PM2.5_today_for_baseline']
expected_baseline = reg_pred['target_date'].sub(pd.Timedelta(days=1)).map(lookup).to_numpy()
assert np.allclose(reg_pred['Persistence'].to_numpy(), expected_baseline, equal_nan=True)
assert reg_pred['target_date'].is_monotonic_increasing and reg_pred['target_date'].is_unique
print('Verified: persistence values align with the preceding day for all saved test targets.')
print('Test rows:', len(reg_pred), '| target dates:', reg_pred.target_date.min().date(), 'to', reg_pred.target_date.max().date())

## 2. Predicted versus actual PM2.5 and error analysis

The next plots show how the best model by RMSE compares with actual observations. The largest absolute-error dates are listed to make model failure visible rather than hiding it behind one average score.

In [ ]:
best_model = reg_metrics.iloc[0]['model']
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(reg_pred.target_date, reg_pred['actual_PM2.5'], label='Actual PM2.5', linewidth=1.6)
ax.plot(reg_pred.target_date, reg_pred[best_model], label=f'{best_model} prediction', linewidth=1.2)
ax.set(title=f'Delhi next-day PM2.5: actual vs {best_model}', xlabel='Target date', ylabel='PM2.5 (µg/m³)')
ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / 'final_regression_actual_vs_predicted.png', dpi=160); plt.show()

error_table = reg_pred[['target_date','actual_PM2.5',best_model]].copy()
error_table['absolute_error'] = (error_table['actual_PM2.5'] - error_table[best_model]).abs()
error_table.sort_values('absolute_error', ascending=False).head(15).to_csv(RESULTS / 'largest_regression_errors.csv', index=False)
error_table.sort_values('absolute_error', ascending=False).head(10)

## 3. Classification evaluation

Target: next-day `AQI_Bucket`. We preserve the chronological split and use class weights to address imbalance; the test data is not oversampled. Accuracy is reported but is not the main selection metric because a model can perform well on common buckets and poorly on rare buckets. Macro-F1 weights each class equally.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

TARGET_DATE = features_df['Date'] + pd.Timedelta(days=1)
train_mask = TARGET_DATE < pd.Timestamp('2020-01-01')
test_mask = (TARGET_DATE >= pd.Timestamp('2020-01-01')) & (TARGET_DATE <= pd.Timestamp('2020-07-01'))
excluded = {'Date','PM2.5_tomorrow','AQI_Bucket_tomorrow','PM2.5_today_for_baseline'}
Xcols = [c for c in features_df.columns if c not in excluded]
cls_valid = features_df['AQI_Bucket_tomorrow'].notna()
train = train_mask & cls_valid
test = test_mask & cls_valid
X_train, X_test = features_df.loc[train, Xcols], features_df.loc[test, Xcols]
y_train = features_df.loc[train, 'AQI_Bucket_tomorrow'].astype(str)
y_test = features_df.loc[test, 'AQI_Bucket_tomorrow'].astype(str)
print('Classification train/test rows:', len(y_train), len(y_test))
print('Training bucket counts: ', y_train.value_counts().to_string())
print('Test bucket counts: ', y_test.value_counts().to_string())

classifiers = {
    'Logistic Regression': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler()), ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))]),
    'Decision Tree': Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=42))]),
    'Random Forest Classifier': Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced_subsample', random_state=42, n_jobs=-1))])
}
class_labels = sorted(set(y_train) | set(y_test))
cls_rows, class_predictions = [], {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    class_predictions[name] = pred
    report = classification_report(y_test, pred, labels=class_labels, zero_division=0, output_dict=True)
    pd.DataFrame(report).T.to_csv(RESULTS / f'{name.lower().replace(" ","_")}_per_class_final.csv')
    cm = confusion_matrix(y_test, pred, labels=class_labels)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels).plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f'{name} — test confusion matrix'); fig.tight_layout()
    fig.savefig(FIGURES / f'{name.lower().replace(" ","_")}_confusion_final.png', dpi=160); plt.show()
    probs = clf.predict_proba(X_test)
    auc_by_class = {}
    for j, label in enumerate(clf.classes_):
        binary = (y_test.to_numpy() == label).astype(int)
        if len(np.unique(binary)) == 2:
            auc_by_class[label] = roc_auc_score(binary, probs[:, j])
    cls_rows.append({
        'task':'classification','model':name,'test_rows':len(y_test),
        'accuracy':accuracy_score(y_test,pred),
        'macro_F1':f1_score(y_test,pred,labels=class_labels,average='macro',zero_division=0),
        'macro_OvR_AUC_available_classes':float(np.mean(list(auc_by_class.values()))) if auc_by_class else np.nan,
        'AUC_classes_evaluated':len(auc_by_class)
    })
    print(name, classification_report(y_test,pred,labels=class_labels,zero_division=0))
cls_metrics = pd.DataFrame(cls_rows).sort_values('macro_F1',ascending=False)
cls_metrics.to_csv(RESULTS / 'classification_metrics_final.csv', index=False)
cls_metrics.round(4)

In [ ]:
# One-vs-rest ROC curves. Classes with no positive or no negative test examples cannot have a defined ROC curve.
fig, ax = plt.subplots(figsize=(8, 6))
for name, clf in classifiers.items():
    probs = clf.predict_proba(X_test)
    for j, label in enumerate(clf.classes_):
        binary = (y_test.to_numpy() == label).astype(int)
        if len(np.unique(binary)) == 2:
            fpr, tpr, _ = roc_curve(binary, probs[:, j])
            auc_val = roc_auc_score(binary, probs[:, j])
            ax.plot(fpr, tpr, label=f'{name} — {label} (AUC={auc_val:.2f})', alpha=0.7)
ax.plot([0,1],[0,1], linestyle='--', linewidth=1, label='Chance')
ax.set(title='One-vs-rest ROC curves — Delhi AQI bucket', xlabel='False positive rate', ylabel='True positive rate')
ax.legend(fontsize=7, loc='lower right'); fig.tight_layout(); fig.savefig(FIGURES / 'classification_ovr_roc_final.png', dpi=160); plt.show()

In [ ]:
# Combine final regression and classification metrics into the required single table.
metrics_final = pd.concat([reg_metrics, cls_metrics], ignore_index=True, sort=False)
metrics_final.to_csv(RESULTS / 'metrics.csv', index=False)
print('Saved final tables and figures to', RESULTS.resolve())
print('Regression conclusion:')
for _, row in reg_metrics.iterrows():
    if row['model'] == 'Persistence': continue
    print(f"{row['model']}: RMSE {row['RMSE']:.3f} vs persistence {base.RMSE:.3f}; RMSE improvement {row['RMSE_improvement_vs_persistence_pct']:.2f}%")
print('Classification best by macro-F1:', cls_metrics.iloc[0]['model'], f"macro-F1={cls_metrics.iloc[0]['macro_F1']:.3f}")